## Setup

In [1]:
import os, time, platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import onnx
import onnxruntime as ort
from onnxruntime.quantization import CalibrationDataReader, QuantFormat, QuantType, quantize_static
from onnxruntime.quantization.shape_inference import quant_pre_process

from inference import EMOTIONS, load_model, preprocess

THREADS = min(4, os.cpu_count())  # same CPU budget for every variant, so the comparison is fair
torch.set_num_threads(THREADS)

DATA_CSV = Path("data/icml_face_data.csv")
WEIGHTS_DIR = Path("weights")
FP32_ONNX = WEIGHTS_DIR / "resnet18_fer_fp32.onnx"
PREP_ONNX = WEIGHTS_DIR / "resnet18_fer_fp32_prep.onnx"
INT8_ONNX = WEIGHTS_DIR / "resnet18_fer_int8.onnx"

print(f"CPU: {platform.processor()} | cores: {os.cpu_count()} | threads used: {THREADS}")
print(f"torch {torch.__version__} | onnx {onnx.__version__} | onnxruntime {ort.__version__}")

CPU: Intel64 Family 6 Model 154 Stepping 4, GenuineIntel | cores: 12 | threads used: 4
torch 2.11.0+cpu | onnx 1.22.0 | onnxruntime 1.30.0


## Data

In [2]:
df = pd.read_csv(DATA_CSV)
df.columns = df.columns.str.strip()
df["Usage"] = df["Usage"].str.strip()

def to_images(pixels):
    return np.array([np.array(p.split(), dtype=np.uint8) for p in pixels]).reshape(-1, 48, 48)

test_df = df[df["Usage"] == "PrivateTest"]
calib_df = df[df["Usage"] == "Training"].sample(n=300, random_state=42)

X_test = np.concatenate([preprocess(img).numpy() for img in to_images(test_df["pixels"])])  # [3589, 3, 96, 96]
y_test = test_df["emotion"].values
X_calib = [preprocess(img).numpy() for img in to_images(calib_df["pixels"])]                 # 300 x [1, 3, 96, 96]
del df
print("test:", X_test.shape, "| calibration images:", len(X_calib))

test: (3589, 3, 96, 96) | calibration images: 300


## Measurement helpers

In [3]:
def size_mb(path):
    return os.path.getsize(path) / 1e6

def evaluate(run):
    """Test accuracy, feeding one image at a time like the app does. Returns (accuracy, predictions)."""
    preds = np.array([run(X_test[i:i + 1]).argmax() for i in range(len(X_test))])
    return (preds == y_test).mean(), preds

def latency_ms(run, warmup=20, runs=200):
    """Median and 95th-percentile time for one face, measured after warm-up runs."""
    x = X_test[:1]
    for _ in range(warmup):
        run(x)
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        run(x)
        times.append((time.perf_counter() - start) * 1000)
    return np.median(times), np.percentile(times, 95)

results, predictions = [], {}

def benchmark(name, run, path):
    acc, preds = evaluate(run)
    median, p95 = latency_ms(run)
    predictions[name] = preds
    results.append({"variant": name, "size_MB": size_mb(path), "test_acc": acc,
                    "latency_ms_median": median, "latency_ms_p95": p95})
    print(f"{name}: acc {acc:.4f} | {size_mb(path):.1f} MB | {median:.2f} ms (p95 {p95:.2f} ms)")

## 1. Baseline: PyTorch FP32

In [4]:
model = load_model()  # CPU, eval mode

@torch.no_grad()
def run_torch(x):
    return model(torch.from_numpy(x)).numpy()

benchmark("PyTorch FP32", run_torch, WEIGHTS_DIR / "ResNet18-pretrained.pth")

PyTorch FP32: acc 0.6851 | 44.8 MB | 68.47 ms (p95 68.95 ms)


## 2. Export to ONNX and run with ONNX Runtime (FP32)

In [5]:
torch.onnx.export(model, (torch.from_numpy(X_test[:1]),), FP32_ONNX,
                  input_names=["input"], output_names=["logits"],
                  dynamo=True, external_data=False)
exported = onnx.load(FP32_ONNX)
onnx.checker.check_model(exported)
print("IR version:", exported.ir_version, "| opsets:", {o.domain or "ai.onnx": o.version for o in exported.opset_import})

def make_session(path):
    options = ort.SessionOptions()
    options.intra_op_num_threads = THREADS
    session = ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name
    return lambda x: session.run(None, {input_name: x})[0]

run_onnx_fp32 = make_session(FP32_ONNX)
max_diff = max(np.abs(run_torch(X_test[i:i + 1]) - run_onnx_fp32(X_test[i:i + 1])).max() for i in range(20))
print(f"Max logit difference PyTorch vs ONNX on 20 images: {max_diff:.2e}")  # expect around 1e-5

benchmark("ONNX Runtime FP32", run_onnx_fp32, FP32_ONNX)

[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\Ako\AppData\Local\Programs\Python\Python313\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
IR version: 10 | opsets: {'ai.onnx': 20}
Max logit difference PyTorch vs ONNX on 20 images: 2.86e-06
ONNX Runtime FP32: acc 0.6851 | 44.8 MB | 57.48 ms (p95 57.77 ms)


### 2b. Same ONNX model, denormals flushed to zero

In [6]:
options = ort.SessionOptions()
options.intra_op_num_threads = THREADS
options.add_session_config_entry("session.set_denormal_as_zero", "1")
session_ftz = ort.InferenceSession(str(FP32_ONNX), options, providers=["CPUExecutionProvider"])
name_ftz = session_ftz.get_inputs()[0].name
benchmark("ONNX Runtime FP32, denormals flushed",
          lambda x: session_ftz.run(None, {name_ftz: x})[0], FP32_ONNX)

ONNX Runtime FP32, denormals flushed: acc 0.6851 | 44.8 MB | 18.06 ms (p95 18.30 ms)


## 3. Static INT8 quantization

In [7]:
quant_pre_process(str(FP32_ONNX), str(PREP_ONNX))

class FERCalibrationReader(CalibrationDataReader):
    """Feeds the calibration images to the quantizer, one at a time."""
    def __init__(self, images, input_name):
        self.batches = iter([{input_name: img} for img in images])

    def get_next(self):
        return next(self.batches, None)

input_name = exported.graph.input[0].name
quantize_static(str(PREP_ONNX), str(INT8_ONNX), FERCalibrationReader(X_calib, input_name),
                quant_format=QuantFormat.QDQ, per_channel=True,
                activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)

run_onnx_int8 = make_session(INT8_ONNX)
benchmark("ONNX Runtime INT8 (static)", run_onnx_int8, INT8_ONNX)

ONNX Runtime INT8 (static): acc 0.6840 | 11.3 MB | 3.00 ms (p95 3.08 ms)


## Results

In [8]:
table = pd.DataFrame(results).set_index("variant")
base = table.loc["PyTorch FP32"]
table["smaller_x"] = base["size_MB"] / table["size_MB"]
table["faster_x"] = base["latency_ms_median"] / table["latency_ms_median"]
table["acc_change_pts"] = (table["test_acc"] - base["test_acc"]) * 100

agreement = (predictions["ONNX Runtime INT8 (static)"] == predictions["PyTorch FP32"]).mean()
print(f"INT8 makes the same prediction as PyTorch FP32 on {agreement:.1%} of test images")
table.round(3)

INT8 makes the same prediction as PyTorch FP32 on 98.7% of test images


,size_MB,test_acc,latency_ms_median,latency_ms_p95,smaller_x,faster_x,acc_change_pts
variant,,,,,,,
PyTorch FP32,44.801,0.685,68.474,68.949,1.000,1.000,0.000
ONNX Runtime FP32,44.793,0.685,57.476,57.770,1.000,1.191,0.000
"ONNX Runtime FP32, denormals flushed",44.793,0.685,18.062,18.300,1.000,3.791,0.000
ONNX Runtime INT8 (static),11.344,0.684,2.998,3.084,3.949,22.844,-0.111


In [9]:
from sklearn.metrics import recall_score

per_class = pd.DataFrame(
    {name: recall_score(y_test, preds, average=None, labels=range(len(EMOTIONS)))
     for name, preds in predictions.items()},
    index=EMOTIONS,
)
per_class.round(3)

,PyTorch FP32,ONNX Runtime FP32,"ONNX Runtime FP32, denormals flushed",ONNX Runtime INT8 (static)
Angry,0.599,0.599,0.599,0.613
Disgust,0.545,0.545,0.545,0.527
Fear,0.434,0.434,0.434,0.432
Happy,0.870,0.870,0.870,0.869
Sad,0.557,0.557,0.557,0.556
Surprise,0.796,0.796,0.796,0.793
Neutral,0.765,0.765,0.765,0.756
